# 00 - Project Setup and Data Audit

## Purpose
This notebook checks that the seven Home Credit data files are available and ready for analysis. It only reads the raw files and does not change them.

## Inputs
Seven CSV files in `data/raw/`.

## Outputs
- `reports/audits/environment.json`
- `reports/audits/file_inventory.csv`
- `reports/audits/schema_inventory.csv`
- `reports/audits/relationship_audit.csv` (when all files are available)

## Important points
- The raw files will not be changed in this notebook.
- `TARGET` will not be used to create features from the historical tables.
- Historical tables will later be summarized to one row per applicant before merging.
- Model preprocessing will be fitted using training data only.

## 1. Import libraries and set the random seed

In [3]:
import json
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Random seed: {RANDOM_SEED}")

Python: 3.12.2
pandas: 3.0.3
NumPy: 2.2.6
Random seed: 42


## 2. Set the project folders
Relative paths are used so the notebook can run on another computer without changing a personal file path.

In [5]:
working_directory = Path.cwd().resolve()
PROJECT_ROOT = working_directory.parent if working_directory.name == "notebooks" else working_directory
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
AUDITS = REPORTS / "audits"
FIGURES = REPORTS / "figures"
MODELS = PROJECT_ROOT / "models"

for directory in (DATA_RAW, DATA_INTERIM, DATA_PROCESSED, AUDITS, FIGURES, MODELS):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw-data directory: {DATA_RAW}")

Project root: /Users/taranveersingh/A-MRP29JULY
Raw-data directory: /Users/taranveersingh/A-MRP29JULY/data/raw


## 3. List the datasets
The table below records each filename, its main key, and what one row represents.

In [7]:
DATASETS = {
    "application": {"filename": "application_train.csv", "keys": ["SK_ID_CURR"], "level": "applicant"},
    "bureau": {"filename": "bureau.csv", "keys": ["SK_ID_BUREAU"], "level": "external credit"},
    "bureau_balance": {"filename": "bureau_balance.csv", "keys": ["SK_ID_BUREAU", "MONTHS_BALANCE"], "level": "external credit-month"},
    "previous_application": {"filename": "previous_application.csv", "keys": ["SK_ID_PREV"], "level": "previous application"},
    "installments": {"filename": "installments_payments.csv", "keys": ["SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER"], "level": "payment event"},
    "credit_card": {"filename": "credit_card_balance.csv", "keys": ["SK_ID_PREV", "MONTHS_BALANCE"], "level": "credit card-month"},
    "pos_cash": {"filename": "POS_CASH_balance.csv", "keys": ["SK_ID_PREV", "MONTHS_BALANCE"], "level": "POS loan-month"},
}

EXPECTED_COLUMNS = {
    "application": {"SK_ID_CURR", "TARGET"},
    "bureau": {"SK_ID_CURR", "SK_ID_BUREAU"},
    "bureau_balance": {"SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"},
    "previous_application": {"SK_ID_CURR", "SK_ID_PREV"},
    "installments": {"SK_ID_CURR", "SK_ID_PREV"},
    "credit_card": {"SK_ID_CURR", "SK_ID_PREV", "MONTHS_BALANCE"},
    "pos_cash": {"SK_ID_CURR", "SK_ID_PREV", "MONTHS_BALANCE"},
}

pd.DataFrame(DATASETS).T

,filename,keys,level
application,application_train.csv,[SK_ID_CURR],applicant
bureau,bureau.csv,[SK_ID_BUREAU],external credit
bureau_balance,bureau_balance.csv,"[SK_ID_BUREAU, MONTHS_BALANCE]",external credit-month
previous_application,previous_application.csv,[SK_ID_PREV],previous application
installments,installments_payments.csv,"[SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTA...",payment event
credit_card,credit_card_balance.csv,"[SK_ID_PREV, MONTHS_BALANCE]",credit card-month
pos_cash,POS_CASH_balance.csv,"[SK_ID_PREV, MONTHS_BALANCE]",POS loan-month


## 4. Record the computer environment
Saving the main software versions makes the project easier to reproduce.

In [9]:
def available_memory_gb():
    try:
        page_size = os.sysconf("SC_PAGE_SIZE")
        available_pages = os.sysconf("SC_AVPHYS_PAGES")
        return round(page_size * available_pages / 1024**3, 2)
    except (AttributeError, OSError, ValueError):
        return None

environment = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "logical_cpu_count": os.cpu_count(),
    "available_memory_gb_at_audit": available_memory_gb(),
    "random_seed": RANDOM_SEED,
}

with (AUDITS / "environment.json").open("w", encoding="utf-8") as file:
    json.dump(environment, file, indent=2)

pd.Series(environment, name="value").to_frame()

,value
created_utc,2026-07-29T19:36:24.408159+00:00
project_root,/Users/taranveersingh/A-MRP29JULY
platform,macOS-26.5.2-arm64-arm-64bit
machine,arm64
processor,arm
python,3.12.2
pandas,3.0.3
numpy,2.2.6
logical_cpu_count,8
available_memory_gb_at_audit,None


## 5. Check that every raw file is available
The original Kaggle CSV files should be placed in `data/raw/` without changing their filenames or columns.

In [11]:
inventory_records = []
for dataset_name, metadata in DATASETS.items():
    path = DATA_RAW / metadata["filename"]
    exists = path.is_file()
    inventory_records.append({
        "dataset": dataset_name,
        "filename": metadata["filename"],
        "exists": exists,
        "size_mb": round(path.stat().st_size / 1024**2, 2) if exists else np.nan,
        "relational_level": metadata["level"],
    })

file_inventory = pd.DataFrame(inventory_records)
file_inventory.to_csv(AUDITS / "file_inventory.csv", index=False)
display(file_inventory)

missing_files = file_inventory.loc[~file_inventory["exists"], "filename"].tolist()
if missing_files:
    print("ACTION REQUIRED - place these files in data/raw/:")
    for filename in missing_files:
        print(f"  - {filename}")
else:
    print("PASS - all seven raw files are available.")

,dataset,filename,exists,size_mb,relational_level
0,application,application_train.csv,True,158.440,applicant
1,bureau,bureau.csv,True,162.140,external credit
2,bureau_balance,bureau_balance.csv,True,358.190,external credit-month
3,previous_application,previous_application.csv,True,386.210,previous application
4,installments,installments_payments.csv,True,689.620,payment event
5,credit_card,credit_card_balance.csv,True,404.910,credit card-month
6,pos_cash,POS_CASH_balance.csv,True,374.510,POS loan-month


PASS - all seven raw files are available.


## 6. Check columns using a sample
Only the column names and the first 5,000 rows are checked here. Detailed missing-value, outlier and distribution analysis will be completed in the EDA notebooks.

In [13]:
schema_records = []
SAMPLE_ROWS = 5_000

for dataset_name, metadata in DATASETS.items():
    path = DATA_RAW / metadata["filename"]
    if not path.is_file():
        continue

    header = pd.read_csv(path, nrows=0)
    sample = pd.read_csv(path, nrows=SAMPLE_ROWS)
    actual_columns = set(header.columns)
    required_columns = EXPECTED_COLUMNS[dataset_name]

    schema_records.append({
        "dataset": dataset_name,
        "column_count": len(header.columns),
        "duplicate_column_names": int(header.columns.duplicated().sum()),
        "missing_required_columns": ", ".join(sorted(required_columns - actual_columns)),
        "sample_rows_read": len(sample),
        "sample_duplicate_rows": int(sample.duplicated().sum()),
        "sample_memory_mb": round(sample.memory_usage(deep=True).sum() / 1024**2, 2),
    })

schema_inventory = pd.DataFrame(schema_records)
schema_inventory.to_csv(AUDITS / "schema_inventory.csv", index=False)
display(schema_inventory)

if not schema_inventory.empty:
    failed = schema_inventory[
        (schema_inventory["duplicate_column_names"] > 0)
        | schema_inventory["missing_required_columns"].ne("")
    ]
    print("PASS - sampled schemas contain the required keys." if failed.empty else "REVIEW - schema problems were detected.")
else:
    print("Schema audit skipped because no raw files are available.")

,dataset,column_count,duplicate_column_names,missing_required_columns,sample_rows_read,sample_duplicate_rows,sample_memory_mb
0,application,122,0,,5000,0,5.290
1,bureau,17,0,,5000,0,0.790
2,bureau_balance,3,0,,5000,0,0.120
3,previous_application,37,0,,5000,0,2.020
4,installments,8,0,,5000,0,0.310
5,credit_card,23,0,,5000,0,0.910
6,pos_cash,8,0,,5000,0,0.330


PASS - sampled schemas contain the required keys.


## 7. Check IDs and table relationships
This section checks how the historical records connect to `application_train.csv`. The historical files also contain Kaggle test applicants, so IDs not found in `application_train.csv` are reported separately and are not automatically treated as data errors. Only ID columns are loaded to keep memory use reasonable. Possible duplicate instalment records are reported but not removed here because split payments may be valid.

In [15]:
relationship_records = []

if not missing_files:
    app_ids = pd.read_csv(DATA_RAW / DATASETS["application"]["filename"], usecols=["SK_ID_CURR"])
    app_id_set = set(app_ids["SK_ID_CURR"])

    relationship_records.append({
        "dataset": "application",
        "rows": len(app_ids),
        "unique_applicants": app_ids["SK_ID_CURR"].nunique(),
        "applicant_coverage_pct": 100.0,
        "ids_not_in_application_train": 0,
        "duplicate_declared_keys": int(app_ids.duplicated(["SK_ID_CURR"]).sum()),
    })

    for dataset_name in ("bureau", "previous_application", "installments", "credit_card", "pos_cash"):
        metadata = DATASETS[dataset_name]
        usecols = list(dict.fromkeys(["SK_ID_CURR", *metadata["keys"]]))
        ids = pd.read_csv(DATA_RAW / metadata["filename"], usecols=usecols)
        dataset_app_ids = set(ids["SK_ID_CURR"].dropna().astype(int))
        relationship_records.append({
            "dataset": dataset_name,
            "rows": len(ids),
            "unique_applicants": ids["SK_ID_CURR"].nunique(),
            "applicant_coverage_pct": round(100 * len(dataset_app_ids & app_id_set) / len(app_id_set), 3),
            "ids_not_in_application_train": len(dataset_app_ids - app_id_set),
            "duplicate_declared_keys": int(ids.duplicated(metadata["keys"]).sum()),
        })
        del ids

    bureau_ids = pd.read_csv(DATA_RAW / DATASETS["bureau"]["filename"], usecols=["SK_ID_BUREAU", "SK_ID_CURR"])
    bureau_balance_ids = pd.read_csv(DATA_RAW / DATASETS["bureau_balance"]["filename"], usecols=["SK_ID_BUREAU", "MONTHS_BALANCE"])
    bureau_id_set = set(bureau_ids["SK_ID_BUREAU"])
    balance_bureau_set = set(bureau_balance_ids["SK_ID_BUREAU"])
    covered_applicants = bureau_ids.loc[
        bureau_ids["SK_ID_BUREAU"].isin(balance_bureau_set)
        & bureau_ids["SK_ID_CURR"].isin(app_id_set),
        "SK_ID_CURR",
    ].nunique()
    relationship_records.append({
        "dataset": "bureau_balance",
        "rows": len(bureau_balance_ids),
        "unique_applicants": covered_applicants,
        "applicant_coverage_pct": round(100 * covered_applicants / len(app_id_set), 3),
        "ids_not_in_application_train": np.nan,
        "duplicate_declared_keys": int(bureau_balance_ids.duplicated(DATASETS["bureau_balance"]["keys"]).sum()),
        "bureau_ids_not_in_bureau": len(balance_bureau_set - bureau_id_set),
    })

    relationship_audit = pd.DataFrame(relationship_records)
    relationship_audit.to_csv(AUDITS / "relationship_audit.csv", index=False)
    display(relationship_audit)
else:
    relationship_audit = pd.DataFrame()
    print("Relationship audit skipped until all seven raw files are available.")

,dataset,rows,unique_applicants,applicant_coverage_pct,ids_not_in_application_train,duplicate_declared_keys,bureau_ids_not_in_bureau
0,application,307511,307511,100.000,0.000,0,NaN
1,bureau,1716428,305811,85.685,"42,320.000",0,NaN
2,previous_application,1670214,338857,94.649,"47,800.000",0,NaN
3,installments,13605401,339587,94.840,"47,944.000",653483,NaN
4,credit_card,3840312,103558,28.261,"16,653.000",0,NaN
5,pos_cash,10001358,337252,94.125,"47,808.000",0,NaN
6,bureau_balance,27299925,92231,29.993,NaN,0,"43,041.000"


## 8. Final checks

In [17]:
checks = {
    "all_raw_files_available": not missing_files,
    "all_required_columns_available": (
        not schema_inventory.empty
        and schema_inventory["missing_required_columns"].eq("").all()
    ),
    "no_duplicate_column_names": (
        not schema_inventory.empty
        and schema_inventory["duplicate_column_names"].eq(0).all()
    ),
}

readiness = pd.Series(checks, name="passed").to_frame()
display(readiness)

if all(checks.values()):
    print("READY - proceed to 01_application_eda.ipynb.")
else:
    print("NOT READY - resolve the failed checks above before table-level EDA.")

,passed
all_raw_files_available,True
all_required_columns_available,True
no_duplicate_column_names,True


READY - proceed to 01_application_eda.ipynb.


## Conclusion
This notebook confirms whether the raw files, important columns and table relationships are available. No cleaning decision is made here.

**Next notebook:** `01_application_eda.ipynb` will examine missing values, unusual values, duplicates, distributions and relationships with `TARGET` in the main application data.